# No2 Prediction of Facilities

## About this Notebook

### Objective

Generate predicted NO₂ concentrations for EPA facilities using the trained fusion model and satellite-derived inputs.

This notebook applies the trained Sentinel-2, Sentinel-5P, or Fusion model to facility-level data in order to estimate atmospheric NO₂ concentrations for emissions verification and downstream analysis.

---

### Inputs

- Cleaned facility dataset (from `CleanFacilityData`)
- Sentinel-2 image chips aligned to facilities
- Sentinel-5P NO₂ raster patches aligned to facilities
- Trained model weights (.pt files)
- Precomputed normalization percentiles

Primary data bucket:
`gs://msads-mba-capstone-team-1/`

Models loaded from:
`gs://msads-mba-capstone-team-1/models/`

---

### Outputs

- Predicted log(NO₂) values for each facility
- Back-transformed NO₂ predictions (if applicable)
- Merged facility dataset with predicted NO₂
- Exported results saved to GCS

Example output path:
`gs://msads-mba-capstone-team-1/Predictions/facility_predictions.csv`

These outputs are used for emissions verification, anomaly detection, and environmental justice analysis.

---

### Workflow Overview

1. Load trained model weights  
2. Initialize facility-level dataset and dataloaders  
3. Generate predictions for each facility  
4. Optionally transform predictions back from log scale  
5. Merge predictions with facility metadata  
6. Export results to GCS  

---

### Key Notes

- The target variable during training was log-transformed NO₂ (`no2_log`).
- Ensure consistent normalization percentiles are used at inference time.
- Model weights must match the architecture defined in `fusionmodels.py`.
- Inference should be performed with `model.eval()` and gradients disabled.
- Predictions should be reproducible using fixed seeds and consistent splits.

---

### Pipeline Position

CleanFacilityData  
→ ExportS2SensorData / ExportS5SensorData  
→ FusionNO2Model  
→ PredictFacilityNO2  

This notebook operationalizes the trained model for facility-level emissions estimation.

## Config

In [2]:
CONFIG = {

    # ----------------------------------
    # MODEL
    # ----------------------------------
    "fusion_model_path":
        "gs://msads-mba-capstone-team-1/models/best_FUSION_model_20260301_052916.pt",

    "s2_percentiles_path":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/s2_percentiles.npy",

    "s5_percentiles_path":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/s5_percentiles.npy",
    
    "fusion_model_folder_path":
        "/home/jupyter/msads-mba-capstone-team-1/Notebooks/team_notebooks/PIPELINE",

    # ----------------------------------
    # STATEWIDE RASTERS (TILED)
    # ----------------------------------
    "s2_statewide_base":
        "msads-mba-capstone-team-1/Data/TrainingData/S2_Statewide",

    "s5_statewide_base":
        "msads-mba-capstone-team-1/Data/TrainingData/S5_Statewide",

    # ----------------------------------
    # FACILITY DATA
    # ----------------------------------
    "facility_data_path":
        "gs://msads-mba-capstone-team-1/Data/TrainingData/nei_2021_IL_nox_for_model.csv",

    # ----------------------------------
    # TIME
    # ----------------------------------
    "year": 2021,
    "quarters": ["Q1", "Q2", "Q3", "Q4"],

    # ----------------------------------
    # OUTPUT
    # ----------------------------------
    "prediction_output_path":
     "/msads-mba-capstone-team-1/Data/Predictions/facility_predictions_2021_model_20260220_054518.csv",
}

## Load Fusion Model

In [3]:
import sys

sys.path.insert(0, CONFIG["fusion_model_folder_path"])

from fusionmodels import (
    S2Dataset,
    S2Model,
    S5OnlyDataset,
    S5Model,
    S5FeatureExtractor,
    FusionDataset,
    FusionModel
)

In [2]:
import torch
import gcsfs
import io
import numpy as np


def load_fusion_model(config, device):

    fs = gcsfs.GCSFileSystem()

    # -----------------------------
    # Load normalization percentiles
    # -----------------------------
    with fs.open(config["s2_percentiles_path"], "rb") as f:
        s2_percentiles = np.load(f, allow_pickle=True)

    with fs.open(config["s5_percentiles_path"], "rb") as f:
        s5_percentiles = np.load(f)

    # -----------------------------
    # Build S2 backbone
    # -----------------------------
    s2_model = S2Model().to(device)
    s2_backbone = s2_model.backbone

    # -----------------------------
    # Build S5 backbone
    # -----------------------------
    s5_model = S5Model().to(device)
    s5_backbone = S5FeatureExtractor(s5_model)

    # -----------------------------
    # Build Fusion model
    # -----------------------------
    fusion_model = FusionModel(s2_backbone, s5_backbone).to(device)

    # -----------------------------
    # Load fusion weights
    # -----------------------------
    with fs.open(config["fusion_model_path"], "rb") as f:
        buffer = io.BytesIO(f.read())
        fusion_model.load_state_dict(torch.load(buffer, map_location=device))

    fusion_model.eval()

    print("Fusion model loaded successfully.")

    return fusion_model, s2_percentiles, s5_percentiles

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fusion_model, s2_percentiles, s5_percentiles = load_fusion_model(CONFIG, device)

Fusion model loaded successfully.


## Load Facilities

In [4]:
import pandas as pd
import gcsfs

def load_facilities(config):

    fs = gcsfs.GCSFileSystem()

    with fs.open(config["facility_data_path"], "r") as f:
        df = pd.read_csv(f)

    print(f"Loaded {len(df)} facilities.")

    return df


facilities = load_facilities(CONFIG)

Loaded 2319 facilities.


In [5]:
facilities.head()

,eis facility id,company name,site name,primary naics code,primary naics description,facility source type,site latitude,site longitude,GEOID10,total emissions,log_emissions
0,558811,NaN,Peoples Gas Light & Coke Co,486210,Pipeline Transportation of Natural Gas,NaN,40.284400,-88.415300,170190105003,78.318483,4.373471
1,558911,Na,Polartech a div of Italmatch SC LLC,325613,Surface Active Agent Manufacturing,NaN,41.772942,-87.801655,170318205021,1.006086,0.696186
2,559311,Na,Owens-Corning Corp,324122,Asphalt Shingle and Coating Materials Manufact...,Hot Mix Asphalt Plant,41.784732,-87.818156,170318204001,27.900600,3.363862
3,559511,Na,Kinder Morgan Liquid Terminals LP,493190,Other Warehousing and Storage,NaN,41.767962,-87.831849,170318205011,6.143930,1.966263
4,559711,Na,LCN Closers Division of Schlage Lock Co LLC,332510,Hardware Manufacturing,NaN,41.386988,-89.467945,170119649002,1.370200,0.862974


## Define Functions for S2/S5 Extraction and NO2 Prediction

In [6]:
import rasterio
import numpy as np
from rasterio.windows import from_bounds
from pyproj import Transformer


def extract_chip_from_tiles(tile_paths, lon, lat):

    half = 600  # meters

    chips = []
    chip_transform = None
    chip_crs = None

    for path in tile_paths:

        with rasterio.open(f"/vsigs/{path}") as src:

            transformer = Transformer.from_crs(
                "EPSG:4326",
                src.crs,
                always_xy=True
            )

            x, y = transformer.transform(lon, lat)

            # Define chip bounds
            left   = x - half
            right  = x + half
            bottom = y - half
            top    = y + half

            # Check intersection with tile bounds
            bounds = src.bounds

            intersects = not (
                right < bounds.left or
                left > bounds.right or
                top < bounds.bottom or
                bottom > bounds.top
            )

            if not intersects:
                continue

            window = from_bounds(
                left, bottom, right, top,
                src.transform
            )

            chip_part = src.read(
                window=window,
                boundless=True,
                fill_value=0
            )

            chips.append(chip_part)

            chip_crs = src.crs
            chip_transform = src.window_transform(window)

    if len(chips) == 0:
        return None

    # Merge overlapping pieces
    chip = np.maximum.reduce(chips)

    return chip

In [7]:
# Normalize chips
def normalize_s2(s2_chip, s2_percentiles):

    for b in range(12):
        p1, p99 = s2_percentiles[b]
        s2_chip[b] = (s2_chip[b] - p1) / (p99 - p1 + 1e-8)
        s2_chip[b] = np.clip(s2_chip[b], 0, 1)

    return s2_chip

def normalize_s2(s2_chip, s2_percentiles):

    for b in range(12):
        p1, p99 = s2_percentiles[b]
        s2_chip[b] = (s2_chip[b] - p1) / (p99 - p1 + 1e-8)
        s2_chip[b] = np.clip(s2_chip[b], 0, 1)

    return s2_chip

In [8]:
from tqdm import tqdm
import numpy as np
import torch
import pandas as pd
import gcsfs

def run_full_prediction(CONFIG,
                        fusion_model,
                        device,
                        s2_percentiles,
                        s5_percentiles):

    fs = gcsfs.GCSFileSystem()
    facilities = load_facilities(CONFIG)

    results = []

    for quarter in CONFIG["quarters"]:

        print(f"\n==============================")
        print(f"Processing {CONFIG['year']} {quarter}")
        print("==============================")

        s2_tiles = fs.glob(
            f"{CONFIG['s2_statewide_base']}/S2_Statewide_{CONFIG['year']}{quarter}*.tif"
        )

        s5_tiles = fs.glob(
            f"{CONFIG['s5_statewide_base']}/S5_Statewide_{CONFIG['year']}{quarter}*.tif"
        )

        print("S2 tiles:", len(s2_tiles))
        print("S5 tiles:", len(s5_tiles))

        for _, row in tqdm(facilities.iterrows(),
                           total=len(facilities)):

            lat = row["site latitude"]
            lon = row["site longitude"]


            s2_chip = extract_chip_from_tiles(s2_tiles, lon, lat)
            s5_chip = extract_chip_from_tiles(s5_tiles, lon, lat)

            # Normalize S2
            for b in range(12):
                p1, p99 = s2_percentiles[b]
                s2_chip[b] = (s2_chip[b] - p1) / (p99 - p1 + 1e-8)
                s2_chip[b] = np.clip(s2_chip[b], 0, 1)

            # Normalize S5
            s5_chip = (s5_chip - s5_percentiles[0]) / (
                s5_percentiles[1] - s5_percentiles[0] + 1e-8
            )
            s5_chip = np.clip(s5_chip, 0, 1)

            # To tensor
            s2_tensor = torch.tensor(s2_chip).unsqueeze(0).float().to(device)
            s5_tensor = torch.tensor(s5_chip).unsqueeze(0).float().to(device)

            with torch.no_grad():
                pred_log = fusion_model(s2_tensor, s5_tensor).item()

            results.append({
                "facility_id": row["eis facility id"],
                "quarter": quarter,
                "predicted_no2_log": pred_log,
                "predicted_no2": float(np.exp(pred_log))
            })

    return pd.DataFrame(results)

## Run Predictions

In [9]:
predictions_df = run_full_prediction(
    CONFIG,
    fusion_model,
    device,
    s2_percentiles,
    s5_percentiles
)

predictions_df.head()

Loaded 2319 facilities.

Processing 2021 Q1
S2 tiles: 15
S5 tiles: 6


100%|██████████| 2319/2319 [08:28<00:00,  4.56it/s]



Processing 2021 Q2
S2 tiles: 15
S5 tiles: 6


100%|██████████| 2319/2319 [09:16<00:00,  4.17it/s]



Processing 2021 Q3
S2 tiles: 15
S5 tiles: 6


100%|██████████| 2319/2319 [09:15<00:00,  4.17it/s]



Processing 2021 Q4
S2 tiles: 15
S5 tiles: 6


100%|██████████| 2319/2319 [09:46<00:00,  3.95it/s]


,facility_id,quarter,predicted_no2_log,predicted_no2
0,558811,Q1,2.520177,12.430801
1,558911,Q1,2.732657,15.373687
2,559311,Q1,2.732053,15.364402
3,559511,Q1,2.732022,15.363926
4,559711,Q1,2.520177,12.430801


In [10]:
predictions_df

,facility_id,quarter,predicted_no2_log,predicted_no2
0,558811,Q1,2.520177,12.430801
1,558911,Q1,2.732657,15.373687
2,559311,Q1,2.732053,15.364402
3,559511,Q1,2.732022,15.363926
4,559711,Q1,2.520177,12.430801
...,...,...,...,...
9271,21291511,Q4,2.520177,12.430801
9272,21292111,Q4,2.630099,13.875141
9273,21292711,Q4,2.520177,12.430801
9274,21382411,Q4,2.657861,14.265745


In [11]:
predictions_df.isna().sum()

facility_id            0
quarter                0
predicted_no2_log    128
predicted_no2        128
dtype: int64

In [12]:
import gcsfs

fs = gcsfs.GCSFileSystem()

with fs.open(CONFIG["prediction_output_path"], "w") as f:
    predictions_df.to_csv(f, index=False)

print("Predictions saved.")

Predictions saved.


## Data Checks

In [13]:
missing_ids = (
    predictions_df[predictions_df["predicted_no2_log"].isna()]
    ["facility_id"]
    .unique()
)

len(missing_ids)

32

In [14]:
missing_facilities = facilities[
    facilities["eis facility id"].isin(missing_ids)
].copy()

missing_facilities[["site latitude", "site longitude"]].head()

,site latitude,site longitude
7,41.626625,-87.530043
364,42.442710,-90.557544
584,41.539867,-87.527626
658,37.692124,-88.139318
723,37.016051,-89.177772


In [15]:
print("Latitude range of missing:")
print(missing_facilities["site latitude"].min(),
      missing_facilities["site latitude"].max())

print("Longitude range of missing:")
print(missing_facilities["site longitude"].min(),
      missing_facilities["site longitude"].max())

Latitude range of missing:
37.016051 42.49387
Longitude range of missing:
-91.394748 -87.527626


In [16]:
IL_LAT_MIN = 36.9
IL_LAT_MAX = 42.5
IL_LON_MIN = -91.5
IL_LON_MAX = -87.0

missing_facilities["dist_to_lat_edge"] = np.minimum(
    missing_facilities["site latitude"] - IL_LAT_MIN,
    IL_LAT_MAX - missing_facilities["site latitude"]
)

missing_facilities["dist_to_lon_edge"] = np.minimum(
    missing_facilities["site longitude"] - IL_LON_MIN,
    IL_LON_MAX - missing_facilities["site longitude"]
)

missing_facilities[[
    "dist_to_lat_edge",
    "dist_to_lon_edge"
]].describe()

,dist_to_lat_edge,dist_to_lon_edge
count,32.000000,32.000000
mean,1.045020,1.326683
std,0.783341,0.588113
min,0.006130,0.105252
25%,0.279293,0.942467
50%,0.927111,1.313979
75%,1.791454,1.847047
max,2.392427,2.184679
